In [ ]:
from pathlib import Path

def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise SystemExit("ไม่พบโฟลเดอร์ datasets/ — เปิด JupyterLab จาก repo root (uv run jupyter lab)")

DATA = data_dir()

# Exercise — Module 1: ML Workflow & EDA

**Dataset (โปรดอ่านก่อนใช้งาน):** *Air Quality* — De Vito, S., et al. (2008). UCI Machine Learning Repository. https://doi.org/10.24432/C59K5F (CC BY 4.0) — ไฟล์ `datasets/air_quality_uci.csv`

โจทย์: ทำ EDA เต็มวงจรครั้งแรกด้วยมือของคุณเอง เติมโค้ดในช่อง `# TODO` — คำใบ้อยู่ใต้แต่ละข้อ เฉลยใน `solution.ipynb`

## ข้อ 1 — โหลดข้อมูลให้ถูกตั้งแต่แรก

โหลด `datasets/air_quality_uci.csv` ให้ polars โดย: ตัวคั่น `;` และแท็กค่าหาย `-200` → null
แล้ว print shape

In [ ]:
import polars as pl

# TODO
aq = ...
print(aq.shape)

**คำใบ้:** ดูวิธีเดียวกับ `book.ipynb` ข้อ 2 — `pl.read_csv(path, separator=..., null_values=...)` คาดหวัง shape `(9358, 13)`

## ข้อ 2 — สร้าง timestamp และเรียงตามเวลา

เพิ่มคอลัมน์ `ts` (Datetime จาก Date+Time) แล้วเรียงแถวตาม `ts` ตรวจว่าไม่มี ts เป็น null

In [ ]:
# TODO
aq = ...
n_bad_ts = ...
print(f"rows with bad timestamp: {n_bad_ts}")

**คำใบ้:** `pl.format("{} {}", ...).str.strptime(pl.Datetime, format="%d/%m/%Y %H.%M.%S", strict=False)` แล้วนับ null ด้วย `.null_count()`

## ข้อ 3 — รายงานค่าหาย

สร้างตารางค่าหายต่อคอลัมน์ เรียงจากหายมากไปน้อย แล้วตอบใน markdown: คอลัมน์ไหนหายเกิน 80% (ควรตัดทิ้ง)

In [ ]:
# TODO
nulls = ...
nulls

คำตอบข้อ 3: (พิมพ์ชื่อคอลัมน์ที่หายเกิน 80% ที่นี่)

## ข้อ 4 — ความสัมพันธ์เซนเซอร์ vs ค่าอ้างอิง

คำนวณ Pearson correlation ระหว่างช่องสัญญาณ `PT08.S1(CO)` … `PT08.S5(O3)` ทั้ง 5 กับ `CO(GT)` (เว้นแถวที่มีค่าหาย) แล้วเรียงจากสัมพันธ์แรงสุด
ตอบ: เซนเซอร์ตัวไหนติดลบ เพราะอะไรจึงไม่ควรแปลกใจ

In [ ]:
# TODO
sensors = ["PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)", "PT08.S4(NO2)", "PT08.S5(O3)"]
corrs = ...
print(corrs)

**คำใบ้:** `aq.select(pl.col(s) for s in sensors, "CO(GT)")` — ไม่ได้! ใช้ `aq.select([pl.col(s) for s in sensors] + [pl.col("CO(GT)")]).drop_nulls()` แล้วคำนวณ corr ต่อคู่ด้วย `pl.corr(a, b)`

## ข้อ 5 — ตรวจจุดผิดปกติด้วย boxplot rule

สำหรับคอลัมน์ `T` (อุณหภูมิ): หาแถวที่ค่าอยู่นอกช่วง [Q1 − 1.5·IQR, Q3 + 1.5·IQR] แล้วรายงานว่ามีกี่แถว ค่าน้อยสุด/มากสุดเท่าไร

In [ ]:
# TODO
q1 = ...
q3 = ...
iqr = ...
lo = ...
hi = ...
outliers = ...
print(f"outliers: {len(outliers)}, min={outliers['T'].min()}, max={outliers['T'].max()}")

**คำใบ้:** quantile: `aq["T"].quantile(0.25)`

## ข้อ 6 — แบ่ง train/test ตามเวลา (ไม่สุ่ม!)

เนื่องจากข้อมูลเรียงตามเวลาแล้ว ให้แบ่ง **80% แรกเป็น train, 20% ท้ายเป็น test** ด้วยการตัดตามตำแหน่งแถว (ไม่ใช้ train_test_split แบบสุ่ม)
รายงานวันแรกของ test set ว่าเป็นวันอะไร

In [ ]:
# TODO
n_train = ...
train = ...
test = ...
first_test_day = ...
print(f"train rows: {len(train)}, test rows: {len(test)}, test starts: {first_test_day}")

## สรุป

เสร็จครบ 6 ข้อ แปลว่าคุณทำ EDA วงจรเต็มได้เอง: โหลดให้ถูก → รู้ว่าไรเรียงอะไร → รายงานค่าหาย → หาความสัมพันธ์ → จับ outlier → แบ่งข้อมูลอย่างซื่อสัตย์